# AN-RA Master Operator Notebook

Run this notebook as the Colab control surface for An-Ra.

It is no longer only a training notebook. It is the full operator loop:

| Cell | Purpose |
| --- | --- |
| 1 | Session configuration and component flags |
| 2 | GPU, RAM, Drive, and dependency setup |
| 3 | Clone or update the repository |
| 4 | Select and merge owner training data |
| 5 | Restore artifacts and run preflight checks |
| 6 | Apply feature flags and print the system report |
| 7 | Run training or evaluation |
| 8 | Inspect telemetry and performance scorecard |
| 9 | Sync reports/checkpoints back to Drive |
| 10 | Launch API/UI surfaces |

Operating rule: every serious run should end with a report, telemetry summary, and saved artifacts.


In [ ]:
# CELL 1 - CONFIG
# Edit this cell first. Everything else should be runnable top-to-bottom.

SESSION_MINUTES = 90

# MODEL SIZE
# "25m" -> current architecture (512 embd, 8 layers) - dev / test
# "1b"  -> V2_1B_FRONTIER (1536 embd, 36 layers, 16 heads) - frontier run
MODEL_SIZE = "1b"

MODEL_SIZE_LABEL = {
    "25m": "25M params - fast, dev/test",
    "1b": "1B params - frontier, RTX6000Ada/A100 required",
}.get(MODEL_SIZE, "unknown")
print(f"Model size: {MODEL_SIZE_LABEL}")
OUROBOROS_MINUTES = 10
TRAINING_MODE = "session"  # "status", "session", "train", or "eval"
RUN_TESTS = False           # set True for a slower full verification pass
RUN_REPORT_BEFORE_TRAIN = True
RUN_REPORT_AFTER_TRAIN = True

REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
REPO_PATH = "/content/An-Ra-the-new-AGI"
DRIVE_ANRA_DIR = "/content/drive/MyDrive/AnRa"
DRIVE_V2_DIR = f"{DRIVE_ANRA_DIR}/v2"
DRIVE_V2_CHECKPOINTS = f"{DRIVE_V2_DIR}/checkpoints"
DRIVE_REPORTS = f"{DRIVE_V2_DIR}/reports"
DRIVE_DATASET = f"{DRIVE_ANRA_DIR}/anra_training.txt"
DRIVE_DATASET_LEGACY = f"{DRIVE_ANRA_DIR}/anra_dataset_v6_1.txt"
LOCAL_MERGED_DATA = "/content/anra_merged_data/anra_training.txt"

# Component flags. Leave everything True unless you are doing ablation or debugging.
COMPONENT_FLAGS = {
    "brain": True,
    "tokenizer": True,
    "data_mix": True,
    "training_loop": True,
    "evaluation": True,
    "runtime": True,
    "api_web": True,
    "identity": True,
    "memory": True,
    "phase2_memory": True,
    "goals": True,
    "agent_loop": True,
    "master_system": True,
    "self_improvement": True,
    "self_modification": True,
    "ouroboros": True,
    "ghost_memory": True,
    "symbolic_bridge": True,
    "sovereignty": True,
}

SELECTED_DATASETS = []
TRAINING_EXIT_CODE = None

print("AN-RA SESSION PLAN")
print("=" * 60)
print(f"Mode:              {TRAINING_MODE}")
print(f"Session minutes:   {SESSION_MINUTES}")
print(f"Model size:        {MODEL_SIZE_LABEL}")
print(f"Ouroboros minutes: {OUROBOROS_MINUTES}")
print(f"Run tests:         {RUN_TESTS}")
print(f"Repo:              {REPO_PATH}")
print(f"Drive root:        {DRIVE_ANRA_DIR}")
print("Enabled components:", sum(1 for v in COMPONENT_FLAGS.values() if v), "/", len(COMPONENT_FLAGS))


In [ ]:
# CELL 2 - ENVIRONMENT SETUP
import os
import sys
import subprocess

from google.colab import drive

print("HARDWARE")
print("=" * 60)
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"RAM: {ram.available / 1024**3:.1f} GB free / {ram.total / 1024**3:.1f} GB total")
except Exception as exc:
    print("RAM: unavailable", exc)

try:
    import torch
    print(f"Torch: {torch.__version__}")
    print(f"CUDA:  {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU:   {torch.cuda.get_device_name(0)}")
except Exception as exc:
    print("Torch check failed:", exc)

try:
    raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,temperature.gpu", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    print("nvidia-smi:", raw)
except Exception:
    print("nvidia-smi: unavailable")

print("\nGOOGLE DRIVE")
print("=" * 60)
drive.mount("/content/drive")
for folder in [DRIVE_ANRA_DIR, DRIVE_V2_DIR, DRIVE_V2_CHECKPOINTS, DRIVE_REPORTS, f"{DRIVE_ANRA_DIR}/logs", f"{DRIVE_ANRA_DIR}/identity"]:
    os.makedirs(folder, exist_ok=True)
print("Drive folders ready")

print("\nDEPENDENCIES")
print("=" * 60)
CORE_PKGS = [
    "numpy", "sympy", "psutil", "tqdm", "pytest", "sentencepiece",
    "transformers", "faiss-cpu", "gitpython", "networkx", "fastapi", "uvicorn",
]
for pkg in CORE_PKGS:
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(f"{pkg:<16} {'ok' if result.returncode == 0 else 'failed'}")
print("Environment ready")


In [ ]:
# CELL 3 - REPOSITORY SETUP
import os
import sys
import shutil
import subprocess

print("REPOSITORY")
print("=" * 60)
if os.path.isdir(REPO_PATH):
    print("Repo exists; updating current checkout")
    result = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_PATH, capture_output=True, text=True)
    if result.returncode != 0:
        print("Fast-forward pull failed; recloning for a clean Colab workspace")
        print(result.stderr[-1000:])
        shutil.rmtree(REPO_PATH)
        result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_PATH], capture_output=True, text=True)
else:
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_PATH], capture_output=True, text=True)

if result.returncode != 0:
    raise RuntimeError(f"Repository setup failed:\n{result.stderr[-2000:]}")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
with open("/tmp/anra_repo_path.txt", "w", encoding="utf-8") as f:
    f.write(REPO_PATH)

print("Repo ready:", REPO_PATH)
print("Current commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_PATH)


In [ ]:
# CELL 4 - DATA SELECTOR AND MERGE
import os
import glob
import json

print("DATA DISCOVERY")
print("=" * 60)
all_txt = glob.glob(f"{DRIVE_ANRA_DIR}/**/*.txt", recursive=True)
all_jsonl = glob.glob(f"{DRIVE_ANRA_DIR}/**/*.jsonl", recursive=True)
all_files = sorted(set(all_txt + all_jsonl))

if not all_files:
    print("No .txt or .jsonl files found under Drive/AnRa")
    SELECTED_DATASETS = []
else:
    for i, fpath in enumerate(all_files):
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"[{i:02d}] {os.path.relpath(fpath, DRIVE_ANRA_DIR):<70} {size_mb:8.2f} MB")
    print("\nSelect files by index, comma-separated, or type 'all'. Press Enter for default anra_training.txt.")
    raw = input("Selection: ").strip()
    if raw.lower() == "all":
        indices = list(range(len(all_files)))
    elif raw:
        indices = [int(x.strip()) for x in raw.split(",") if x.strip().isdigit()]
    else:
        indices = []
    SELECTED_DATASETS = [all_files[i] for i in indices if 0 <= i < len(all_files)]
    if not SELECTED_DATASETS and os.path.exists(DRIVE_DATASET):
        SELECTED_DATASETS = [DRIVE_DATASET]

print("\nMERGE")
print("=" * 60)
os.makedirs(os.path.dirname(LOCAL_MERGED_DATA), exist_ok=True)
if SELECTED_DATASETS:
    with open(LOCAL_MERGED_DATA, "w", encoding="utf-8") as out:
        for fpath in SELECTED_DATASETS:
            try:
                if fpath.endswith(".jsonl"):
                    for line in open(fpath, encoding="utf-8", errors="replace"):
                        try:
                            obj = json.loads(line)
                        except Exception:
                            continue
                        for key in ["text", "content", "prompt", "response", "output", "answer"]:
                            value = obj.get(key)
                            if isinstance(value, str) and value.strip():
                                out.write(value.strip() + "\n\n")
                                break
                else:
                    out.write(open(fpath, encoding="utf-8", errors="replace").read().strip() + "\n\n")
                print("merged", os.path.basename(fpath))
            except Exception as exc:
                print("failed", fpath, exc)
    DRIVE_DATASET = LOCAL_MERGED_DATA
    print("Merged dataset:", LOCAL_MERGED_DATA, f"({os.path.getsize(LOCAL_MERGED_DATA) / 1024**2:.2f} MB)")
else:
    print("No selected dataset. Training will use repo bootstrap data if available.")


In [ ]:
# CELL 5 - ARTIFACT RESTORE AND PREFLIGHT
import os
import glob
import shutil
import subprocess
import sys

print("ARTIFACT RESTORE")
print("=" * 60)
local_data = f"{REPO_PATH}/training_data/anra_training.txt"
os.makedirs(os.path.dirname(local_data), exist_ok=True)

source = None
for candidate in [DRIVE_DATASET, DRIVE_DATASET_LEGACY]:
    if candidate and os.path.exists(candidate):
        source = candidate
        break
if source:
    shutil.copy2(source, local_data)
    print("Dataset copied:", source, "->", local_data)
else:
    print("No Drive dataset copied; repo data will be used if present")

for pt in glob.glob(f"{DRIVE_V2_CHECKPOINTS}/*.pt"):
    dst = f"{REPO_PATH}/{os.path.basename(pt)}"
    shutil.copy2(pt, dst)
    print("Checkpoint restored:", os.path.basename(pt))

print("\nPREFLIGHT")
print("=" * 60)
checks = [
    (os.path.exists(f"{REPO_PATH}/anra.py"), "anra.py present"),
    (os.path.exists(f"{REPO_PATH}/anra_paths.py"), "anra_paths.py present"),
    (os.path.exists(f"{REPO_PATH}/runtime/system_registry.py"), "system registry present"),
    (os.path.exists(f"{REPO_PATH}/engine/report.py"), "engineering report module present"),
    (os.path.exists(f"{REPO_PATH}/tokenizer/tokenizer_v3.json"), "tokenizer_v3 present"),
    (os.path.exists(local_data), "training data present"),
]
failed = False
for ok, label in checks:
    print(f"{'PASS' if ok else 'FAIL'} - {label}")
    failed = failed or not ok
if failed:
    raise RuntimeError("Preflight failed. Fix missing artifacts before continuing.")

if MODEL_SIZE == "1b":
    print("\n-- 1B MODEL PREFLIGHT --------------------------------------")
    try:
        import torch
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
        gpu_name = torch.cuda.get_device_properties(0).name
        print(f"  GPU  : {gpu_name}")
        print(f"  VRAM : {vram_gb:.1f} GB")
        if vram_gb < 20:
            raise SystemExit(
                f"\n1B ABORTED: only {vram_gb:.1f}GB VRAM detected.\n"
                f"Minimum 20GB. Use RTX 6000 Ada on io.net ($0.97/hr)."
            )
        elif vram_gb < 40:
            print("  WARNING: batch_size=2 grad_accum=16 will be enforced")
        else:
            print("  VRAM sufficient for full 1B training")
    except SystemExit:
        raise
    except Exception:
        print("  WARNING: GPU check failed - proceeding anyway")

print("\nRegistry smoke:")
subprocess.run([sys.executable, "-c", "from runtime.system_registry import component_registry; print(len(component_registry()), 'components')"], cwd=REPO_PATH)


In [ ]:
# CELL 6 - FEATURE FLAGS AND SYSTEM REPORT
import os
import sys
import subprocess

print("APPLY FEATURE FLAGS")
print("=" * 60)
flag_code = """
from engine.feature_flags import set_flag, load_flags
flags = %r
for name, enabled in flags.items():
    set_flag(name, bool(enabled))
print(load_flags())
""" % COMPONENT_FLAGS
result = subprocess.run([sys.executable, "-c", flag_code], cwd=REPO_PATH, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(result.stderr[-1000:])

if RUN_REPORT_BEFORE_TRAIN:
    print("\nSYSTEM REPORT")
    print("=" * 60)
    process = subprocess.run([sys.executable, "anra.py", "--report"], cwd=REPO_PATH, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(process.stdout[-6000:])
    if process.returncode != 0:
        raise RuntimeError("anra.py --report failed")


In [ ]:
# CELL 7 - TRAINING OR EVALUATION RUN
import os
import sys
import time
import subprocess

print("RUN")
print("=" * 60)
cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode", TRAINING_MODE,
    "--model-size", MODEL_SIZE,
]
if TRAINING_MODE not in {"status", "eval"}:
    cmd.extend([
        "--session-minutes", str(SESSION_MINUTES),
        "--ouroboros_minutes", str(OUROBOROS_MINUTES),
    ])
print("Command:", " ".join(cmd))

env = os.environ.copy()
env["PYTHONPATH"] = REPO_PATH
start = time.time()
process = subprocess.Popen(cmd, cwd=REPO_PATH, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
TRAINING_EXIT_CODE = process.returncode
elapsed = (time.time() - start) / 60
print("
Run finished:", TRAINING_EXIT_CODE, f"elapsed={elapsed:.1f} min")
if TRAINING_EXIT_CODE != 0:
    raise RuntimeError(f"Training/eval command failed with code {TRAINING_EXIT_CODE}")


In [ ]:
# CELL 8 - METRICS
import os
import sys
import subprocess

print("METRICBUS SNAPSHOT")
print("=" * 60)
score_code = """
from engine.metric_bus import get_metric_bus
bus = get_metric_bus()
print(bus.snapshot())
"""
result = subprocess.run([sys.executable, "-c", score_code], cwd=REPO_PATH, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr[-1000:])

if RUN_REPORT_AFTER_TRAIN:
    print("
POST-RUN REPORT")
    print("=" * 60)
    process = subprocess.run([sys.executable, "anra.py", "--report"], cwd=REPO_PATH, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(process.stdout[-6000:])
    if process.returncode != 0:
        raise RuntimeError("post-run report failed")

if RUN_TESTS:
    print("
TESTS")
    print("=" * 60)
    process = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-x", "-q"], cwd=REPO_PATH, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(process.stdout[-8000:])
    if process.returncode != 0:
        raise RuntimeError("tests failed")


In [ ]:
# CELL 9 - POST-SESSION SCORECARD TO DRIVE
import os
import sys
import subprocess

print("POST-SESSION SCORECARD")
print("=" * 60)
post_code = f"""
from agents.supervisor import SupervisorAgent
from engine.metric_bus import get_metric_bus

bus = get_metric_bus()
run_data = bus.finalize()
print(f"Run ID: {{run_data['run_id']}}")
print(f"Duration: {{run_data.get('duration_minutes', 0):.1f}} min")
print(f"Components active: {{list(run_data['components'].keys())}}")
if run_data.get('deltas'):
    print("Deltas vs last run:")
    for comp, d in run_data['deltas'].items():
        print(f"  {{comp}}: {{d}}")

supervisor = SupervisorAgent(model_size={MODEL_SIZE!r})
summary = supervisor.end_session()
supervisor.push_scorecard_to_drive(summary)
print("\nScorecard written. Open Drive -> AnRa/logs/scorecards/ to review.")
"""
result = subprocess.run([sys.executable, "-c", post_code], cwd=REPO_PATH, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr[-2000:])
if result.returncode != 0:
    raise RuntimeError("post-session scorecard failed")


In [ ]:
# CELL 10 - SYNC REPORTS AND CHECKPOINTS TO DRIVE
import os
import glob
import shutil

print("SYNC TO DRIVE")
print("=" * 60)
os.makedirs(DRIVE_REPORTS, exist_ok=True)
os.makedirs(DRIVE_V2_CHECKPOINTS, exist_ok=True)

report_sources = [f"{REPO_PATH}/output/v2/reports", f"{REPO_PATH}/output/v2/eval"]
for folder in report_sources:
    if os.path.isdir(folder):
        for fpath in glob.glob(f"{folder}/*.json"):
            dst = f"{DRIVE_REPORTS}/{os.path.basename(fpath)}"
            shutil.copy2(fpath, dst)
            print("report", os.path.basename(fpath))

for pattern in ["anra_v2_*.pt", "*.pt"]:
    for fpath in glob.glob(f"{REPO_PATH}/{pattern}"):
        dst = f"{DRIVE_V2_CHECKPOINTS}/{os.path.basename(fpath)}"
        try:
            shutil.copy2(fpath, dst)
            print("checkpoint", os.path.basename(fpath))
        except Exception as exc:
            print("checkpoint sync failed", fpath, exc)

print("Drive reports:", DRIVE_REPORTS)
print("Drive checkpoints:", DRIVE_V2_CHECKPOINTS)


In [ ]:
# CELL 10 - API OR UI LAUNCH
# Optional. Run after repo setup. The CLI report/training cells above are the primary operator path.
import os
import sys
import subprocess

REPO_PATH = open("/tmp/anra_repo_path.txt", encoding="utf-8").read().strip()
print("Repo:", REPO_PATH)
print("Available launch options:")
print("1. CLI status:   !python anra.py --status")
print("2. CLI report:   !python anra.py --report")
print("3. FastAPI app:   python app.py or uvicorn app:app --host 0.0.0.0 --port 8000")
print("4. Legacy UI:     run ui/anra_ui_launcher.py if present")

launcher = f"{REPO_PATH}/ui/anra_ui_launcher.py"
if os.path.exists(launcher):
    run_ui = input("Launch legacy UI now? [y/N]: ").strip().lower() == "y"
    if run_ui:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask", "flask-cors"], capture_output=True)
        exec(open(launcher, encoding="utf-8").read())
else:
    print("Legacy UI launcher not found; use anra.py or app.py surfaces.")
